<a href="https://colab.research.google.com/github/Hesgoryr/HRNet_Grasp_Semantic_Segmentation/blob/main/notebooks/colab_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HRNet-W18 Jacquard V2 — Google Colab training

Run all cells top to bottom. The notebook expects a free Colab T4 instance.

**What it does**
1. Clones the repo and installs deps (CUDA torch + imagecodecs).
2. Downloads the dataset zips from your Yandex Disk public folder into `/content/Jacquard_V2_zips/`.
3. Unzips them into `/content/Jacquard_V2/` (~11k object folders).
4. Mounts Google Drive at `/content/drive/` so checkpoints survive session resets.
5. Builds the object-wise split (only first time).
6. Trains with `--resume` if a previous run is found in Drive.

**Storage layout (Colab session-local SSD)**
- `/content/Jacquard_V2_zips/*.zip` — temporary, can be deleted after unpack.
- `/content/Jacquard_V2/<object_id>/...` — the training data.
- `/content/drive/MyDrive/hrnet_runs/<run_name>/` — persistent checkpoints + metrics.csv.

**Sessions reset every ~12 h on free Colab.** When that happens, re-run all cells; download is idempotent and `--resume` picks up from `last.pth` in Drive.

## 0. User-supplied configuration

Edit the next cell before running.

In [ ]:
# ====== EDIT THESE ======
YANDEX_DISK_FOLDER_URL = "https://disk.yandex.ru/d/Je56nUcC9hiHFQ"  # public Yandex Disk share
YANDEX_FOLDER_PATH    = "/Jacquard_V2"  # subfolder *inside* the share that holds the 12 zips; "" if zips are at the root

# Pick which experiment to run.
#   "configs/default.yaml"   — angle mode (19-class mask, baseline). Use this
#                              for the main RGB-D run reported in docs.
#   "configs/multitask.yaml" — Phase 2a multitask (pos+cos2t+sin2t+width) head.
#                              Switch to this if angle mode plateaus below the
#                              0.40 mIoU_fg target.
CONFIG_FILE = "configs/default.yaml"
RUN_NAME = "hrnet_w18_rgbd_angle"  # subfolder under MyDrive/hrnet_runs/ — change when switching CONFIG_FILE so checkpoints don't collide
GITHUB_REPO = "https://github.com/Hesgoryr/HRNet_Grasp_Semantic_Segmentation.git"
GITHUB_BRANCH = "main"
# Training overrides (T4 has 16 GB VRAM — bigger batch than RTX 3060)
IMAGE_SIZE = 384
BATCH_SIZE = 16
ACCUM_STEPS = 1
NUM_WORKERS = 4
EPOCHS = 80
# ========================

## 1. Verify GPU

In [2]:
!nvidia-smi

Sun Apr 26 19:39:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone repo + install missing deps

Colab ships with CUDA torch, numpy, opencv, tifffile, matplotlib, pandas, pyyaml, tqdm already installed. We install ONLY the two packages that aren't preinstalled: `timm` and `imagecodecs`. Do **not** `pip install -r requirements.txt` on Colab — it can upgrade transitive deps of the already-loaded torch and trigger `cannot import name 'nn' from partially initialized module 'torch'` on subsequent `import torch`.

In [3]:
import os, subprocess
REPO_DIR = '/content/HRNet_Grasp_Semantic_Segmentation'
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 -b {GITHUB_BRANCH} {GITHUB_REPO} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch --depth 1 origin {GITHUB_BRANCH} && git reset --hard origin/{GITHUB_BRANCH}
%cd {REPO_DIR}
!pip install -q timm imagecodecs

Cloning into '/content/HRNet_Grasp_Semantic_Segmentation'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 86 (delta 9), reused 68 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 763.84 KiB | 4.83 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/HRNet_Grasp_Semantic_Segmentation
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 78.6 MB/s eta 0:00:00


In [4]:
import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

torch: 2.10.0+cu128 cuda: True Tesla T4


## 3. Mount Google Drive (for persistent checkpoints)

In [5]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = f'/content/drive/MyDrive/hrnet_runs/{RUN_NAME}'
os.makedirs(SAVE_DIR, exist_ok=True)
print('checkpoints will go to:', SAVE_DIR)

Mounted at /content/drive
checkpoints will go to: /content/drive/MyDrive/hrnet_runs/hrnet_w18_rgbd_angle


## 4. Download dataset from Yandex Disk

Lists every file inside the public folder via the public-resources API, then downloads each in turn. Already-present files are skipped, so the cell is safe to re-run after a session reset.

In [ ]:
import os, requests, time

ZIP_DIR = '/content/Jacquard_V2_zips'
os.makedirs(ZIP_DIR, exist_ok=True)
API = 'https://cloud-api.yandex.net/v1/disk/public/resources'

def list_public_folder(public_url, sub_path='', limit=100):
    """Yield (filename, inner_path) for every file inside a public Yandex Disk folder."""
    offset = 0
    while True:
        params = {'public_key': public_url, 'limit': limit, 'offset': offset}
        if sub_path:
            params['path'] = sub_path
        r = requests.get(API, params=params, timeout=60)
        r.raise_for_status()
        items = r.json().get('_embedded', {}).get('items', [])
        if not items:
            return
        for it in items:
            if it['type'] == 'file':
                yield it['name'], it.get('path')
        if len(items) < limit:
            return
        offset += limit

def download_one(public_url, dest, inner_path):
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        print(f'  skip (exists): {os.path.basename(dest)}')
        return
    r = requests.get(API + '/download', params={'public_key': public_url, 'path': inner_path}, timeout=60)
    r.raise_for_status()
    href = r.json()['href']
    print(f'  downloading {os.path.basename(dest)}…')
    with requests.get(href, stream=True, timeout=600) as resp:
        resp.raise_for_status()
        total = int(resp.headers.get('content-length', 0))
        done, t0 = 0, time.time()
        with open(dest + '.part', 'wb') as f:
            for chunk in resp.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    f.write(chunk)
                    done += len(chunk)
                    if total and (time.time() - t0) > 5:
                        pct = 100 * done / total
                        mbs = done / 1e6 / max(time.time() - t0, 1e-3)
                        print(f'    {pct:5.1f}%  {mbs:5.1f} MB/s', end='\r')
        os.rename(dest + '.part', dest)
    print(f'    done ({os.path.getsize(dest)/1e9:.2f} GB)')

files = list(list_public_folder(YANDEX_DISK_FOLDER_URL, YANDEX_FOLDER_PATH))
print(f'found {len(files)} file(s) in {YANDEX_FOLDER_PATH or "<root>"}')
for name, inner_path in files:
    download_one(YANDEX_DISK_FOLDER_URL, os.path.join(ZIP_DIR, name), inner_path)
print('all zips ready in', ZIP_DIR)
!ls -lh {ZIP_DIR}

found 12 file(s) in /Jacquard_V2
  downloading JacquardV2_Dataset_0.zip…


## 5. Unzip into a single flat folder

We extract every archive into `/content/Jacquard_V2/`. Each zip contains a top-level `JacquardV2_Dataset_<N>/` folder which we strip so all object folders end up flat under `/content/Jacquard_V2/`. `prepare_split.py` happily handles either layout (flat or nested-by-shard).

In [ ]:
import zipfile, glob, shutil

DATA_DIR = '/content/Jacquard_V2'
os.makedirs(DATA_DIR, exist_ok=True)

for z in sorted(glob.glob(os.path.join(ZIP_DIR, '*.zip'))):
    print('extracting', os.path.basename(z))
    with zipfile.ZipFile(z) as zf:
        zf.extractall(DATA_DIR)

# Each zip extracts into its own Jacquard*Dataset_<N>/ subfolder; flatten so
# discover_dataset() finds object folders one level below DATA_DIR. The
# Yandex Disk mirror ships 'Jacquard_Dataset_N' while the upstream Jacquard
# V2 release uses 'JacquardV2_Dataset_N' -- match both.
for shard_dir in sorted(
    glob.glob(os.path.join(DATA_DIR, 'JacquardV2_Dataset_*'))
    + glob.glob(os.path.join(DATA_DIR, 'Jacquard_Dataset_*'))
):
    if not os.path.isdir(shard_dir):
        continue
    for entry in os.listdir(shard_dir):
        src = os.path.join(shard_dir, entry)
        dst = os.path.join(DATA_DIR, entry)
        if os.path.exists(dst):
            continue
        shutil.move(src, dst)
    try:
        os.rmdir(shard_dir)
    except OSError:
        pass

print('counting grasps files…')
!find {DATA_DIR} -mindepth 1 -maxdepth 3 -name '*_grasps.txt' | wc -l

## 6. Build object-wise split (idempotent)

In [ ]:
SPLITS_PATH = os.path.join(REPO_DIR, 'splits/jacquard_v2.json')
if not os.path.exists(SPLITS_PATH):
    !python tools/prepare_split.py --root {DATA_DIR} --out {SPLITS_PATH} --val-frac 0.1 --test-frac 0.1 --seed 0
else:
    print('split already exists, reusing:', SPLITS_PATH)

In [ ]:
import shutil, os
DRIVE_SPLITS = '/content/drive/MyDrive/hrnet_runs/splits/jacquard_v2.json'
os.makedirs(os.path.dirname(DRIVE_SPLITS), exist_ok=True)
if not os.path.exists(DRIVE_SPLITS):
    shutil.copy(SPLITS_PATH, DRIVE_SPLITS)
    print(f'Saved split -> {DRIVE_SPLITS}')
else:
    print(f'Drive split already exists: {DRIVE_SPLITS}')

# Проверка: сколько объектов/файлов
import json
d = json.load(open(DRIVE_SPLITS))
print(f'train={len(d["train"])} val={len(d["val"])} test={len(d["test"])}')

Число объектов должно быть 11619 каждый раз

## 7. Train (auto-resume from `last.pth` in Drive if present)

In [ ]:
RESUME_PATH = os.path.join(SAVE_DIR, 'last.pth')
resume_arg = f'--resume {RESUME_PATH}' if os.path.exists(RESUME_PATH) else ''
cmd = (f'python tools/train.py --config {CONFIG_FILE} '
       f'dataset.splits_path={SPLITS_PATH} '
       f'dataset.image_size={IMAGE_SIZE} '
       f'dataset.num_workers={NUM_WORKERS} '
       f'trainer.epochs={EPOCHS} '
       f'trainer.batch_size={BATCH_SIZE} '
       f'trainer.accum_steps={ACCUM_STEPS} '
       f'trainer.save_dir={SAVE_DIR} '
       f'{resume_arg}')
print(cmd)
!{cmd}

## 8. Optional — quick metric plot from `metrics.csv`

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv(os.path.join(SAVE_DIR, 'metrics.csv'))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if 'train_loss' in df.columns:
    axes[0].plot(df['epoch'], df['train_loss'], label='train_loss')
axes[0].set_title('train loss'); axes[0].legend(); axes[0].grid()
# In angle mode `val_miou_fg` is the multi-class mIoU over 18 angle bins.
# In multitask mode it's the binary fg-vs-bg IoU; the directly-comparable
# multi-class metric is logged as `val_miou_fg_ang` (and `val_dice_fg_ang`).
val_cols = [c for c in ['val_miou_fg', 'val_dice_fg', 'val_miou_fg_ang', 'val_dice_fg_ang'] if c in df.columns]
for col in val_cols:
    axes[1].plot(df['epoch'], df[col], label=col)
axes[1].set_title('val metrics'); axes[1].legend(); axes[1].grid()
plt.tight_layout(); plt.show()

In [ ]:
import os, glob
# Путь к чекпоинту в Drive — используйте RUN_NAME с предыдущего angle-run.
SAVE_DIR = '/content/drive/MyDrive/hrnet_runs/hrnet_w18_rgbd_angle'
CKPT = os.path.join(SAVE_DIR, 'best.pth')
SPLITS_PATH = '/content/HRNet_Grasp_Semantic_Segmentation/splits/jacquard_v2.json'
print('checkpoint:', CKPT, 'exists:', os.path.exists(CKPT))
print('splits:', SPLITS_PATH, 'exists:', os.path.exists(SPLITS_PATH))

# eval на val (sanity-check — должно совпадать с metrics.csv epoch 25)
!python tools/eval.py --config configs/default.yaml --checkpoint {CKPT} --split val \
    dataset.splits_path={SPLITS_PATH} \
    dataset.image_size=384 \
    dataset.num_workers=8 \
    trainer.batch_size=48 2>&1 | tail -5

# eval на test — главная цифра в отчёт
!python tools/eval.py --config configs/default.yaml --checkpoint {CKPT} --split test \
    dataset.splits_path={SPLITS_PATH} \
    dataset.image_size=384 \
    dataset.num_workers=8 \
    trainer.batch_size=48 2>&1 | tail -5